### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [34]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:llama-3.3-70b-versatile" , temperature=0)
response=model.invoke("Hello how are you?")
response

AIMessage(content="Hello! I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or tasks you may have. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 40, 'total_tokens': 88, 'completion_time': 0.124137373, 'completion_tokens_details': None, 'prompt_time': 0.001959153, 'prompt_tokens_details': None, 'queue_time': 0.051534286, 'total_time': 0.126096526}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebb77-e334-7d41-a7c4-8185a10f1496-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 48, 'total_tokens': 88})

In [31]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage 

@tool
def get_weather(location:str)-> str:
    """Get the weather for a location"""
    return f"The weather in {location} is sunny."


model_with_tools=model.bind_tools([get_weather])


In [26]:
response = model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': '54qbvs5yf', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.039016597, 'completion_tokens_details': None, 'prompt_time': 0.010674347, 'prompt_tokens_details': None, 'queue_time': 0.05193044, 'total_time': 0.049690944}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019ebb57-654c-7850-a354-dcee74eaa959-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '54qbvs5yf', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops


In [32]:
messages = [
    {"role": "system", "content": "When you receive tool results, use them directly to answer the user's question. Do not say you don't have access to the information."},
    HumanMessage(content="What's the weather in Boston?")
]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

print(messages)
final_response=model_with_tools.invoke(messages)
print(final_response.text)

[{'role': 'system', 'content': "When you receive tool results, use them directly to answer the user's question. Do not say you don't have access to the information."}, HumanMessage(content="What's the weather in Boston?", additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'rjqjrc2tk', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 247, 'total_tokens': 261, 'completion_time': 0.048751862, 'completion_tokens_details': None, 'prompt_time': 0.013173921, 'prompt_tokens_details': None, 'queue_time': 0.050361978, 'total_time': 0.061925783}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ebb77-5baf-7532-903c-940bf3129646-0', tool_calls=[{'name': 'get_weather', '

### Messages

In LangChain, messages are the standard format for representing a conversation with a chat model — each message has a role and content, and LangChain provides typed classes for each role

1. HumanMessage — input from the user.
2. AIMessage — output from the model. Can contain plain text, or tool_calls if the model decided to call a tool (as you saw, with empty content and a tool_calls list).
3. SystemMessage — instructions/context that shape the model's behavior, sent before the conversation starts.
4. ToolMessage — the result of executing a tool, sent back to the model so it can use that result. Must include tool_call_id matching the id from the corresponding tool call in the AIMessage.

### MESSAGE PROMPTS
1. System message — sets the model's behavior/persona/instructions for the whole conversation (e.g., "You are a helpful assistant that only responds in JSON").
2. Human message — the user's input/question to the model.
3. AI message — the model's response. This can be plain text, or it can include tool_calls (when the model decides it needs to call a function instead of answering directly), plus metadata like token usage.
4. Tool message — the result of actually running that tool/function, sent back to the model so it can incorporate that result into its next response. Linked to the AI message's tool call via tool_call_id.